In [ ]:
from emu_renewal.outputs import get_output_dir, get_combined_df, get_df_from_3darray, melt_df_except_first_level
import arviz as az
import pandas as pd
import numpy as np
import seaborn as sns

In [ ]:
country = "Spain"
analysis_times = {
    "no_mob": "20250116_1321",
    "google_nonresi_linear": "20250116_1335",
    "google_nonresi_square": "20250116_1344",
}

In [ ]:
idatas = {k: az.from_netcdf(get_output_dir(country, k, v) / "idata.nc") for k, v in analysis_times.items()}

In [ ]:
# Get the variable process and dispersion values, with samples as first axis (for use as index to dataframe later)
procs = []
disps = []
for a in analysis_times:
    idata = idatas[a]
    proc_vals = np.swapaxes(idata.posterior["proc"].to_numpy(), 0, 1)
    procs.append(get_df_from_3darray(proc_vals, [0, 2, 1]))
    disps.append(pd.DataFrame(np.swapaxes(idata.posterior["dispersion_proc"].to_numpy(), 0, 1)))
multianalysis_proc_df = pd.concat(procs, axis=1, keys=analysis_times.keys())
multianalysis_disp_df = pd.concat(disps, axis=1, keys=analysis_times.keys())

In [ ]:
disp_comparison_df = melt_df_except_first_level(multianalysis_disp_df)
sns.kdeplot(disp_comparison_df, fill=True)

In [ ]:
proc_comparison_df = melt_df_except_first_level(multianalysis_proc_df)
sns.kdeplot(proc_comparison_df, fill=True)